# FairScope: On-Device Model Quantization Pipeline
**Member C:** Model Compression & On-Phone Optimization (SLIIT Research Project J26-IT-371)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanuka001/Fairscope_research/blob/feat/monal/app-shell/models/compressed/quantization_pipeline.ipynb)

---

### Overview
This notebook demonstrates the **Post-Training Quantization (PTQ)** pipeline for edge deployment:
1. Loads the baseline feature extractor (`MobileNetV2` with ImageNet weights, $224 \times 224 \times 3$).
2. Exports the uncompressed **Float32 baseline model** (`baseline_fp32.tflite`).
3. Applies 8-bit integer quantization mapping 32-bit float parameters to 8-bit integers.
4. Exports the **Compressed INT8 model** (`quantized_int8.tflite`).
5. Evaluates and plots model file size reduction, parameter compression, and memory footprint.

In [ ]:
# Step 1: Imports and Environment Verification
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# Step 2: Define Export Paths
output_dir = "models_export"
os.makedirs(output_dir, exist_ok=True)

fp32_path = os.path.join(output_dir, "baseline_fp32.tflite")
int8_path = os.path.join(output_dir, "quantized_int8.tflite")
labels_path = os.path.join(output_dir, "labels.txt")

print(f"Target export directory: {os.path.abspath(output_dir)}")

In [ ]:
# Step 3: Load the Baseline Model
# MobileNetV2 architecture (standard lightweight mobile vision backbone)
IMG_SHAPE = (224, 224, 3)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=True,
    weights='imagenet'
)

base_model.summary()

In [ ]:
# Step 4: Convert and Export Baseline Float32 Model
print("Converting Baseline FP32 Model...")
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(base_model)
tflite_fp32_model = converter_fp32.convert()

with open(fp32_path, "wb") as f:
    f.write(tflite_fp32_model)

fp32_size_mb = os.path.getsize(fp32_path) / (1024 * 1024)
print(f"✅ Baseline FP32 saved to: {fp32_path}")
print(f"Size: {fp32_size_mb:.2f} MB")

In [ ]:
# Step 5: Post-Training INT8 Quantization (PTQ)
# Quantizes weights from 32-bit floating points down to 8-bit integers
print("Converting Quantized INT8 Model...")
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(base_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

# Optional: Representative dataset generator for full integer calibration
def representative_data_gen():
    for _ in range(50):
        # Synthetic or calibration image tensor (1, 224, 224, 3)
        data = np.random.uniform(-1.0, 1.0, size=(1, 224, 224, 3)).astype(np.float32)
        yield [data]

converter_int8.representative_dataset = representative_data_gen

tflite_int8_model = converter_int8.convert()

with open(int8_path, "wb") as f:
    f.write(tflite_int8_model)

int8_size_mb = os.path.getsize(int8_path) / (1024 * 1024)
print(f"✅ Quantized INT8 saved to: {int8_path}")
print(f"Size: {int8_size_mb:.2f} MB")

In [ ]:
# Step 6: Export Stand-in Labels
# Sample labels for stand-in classification
sample_labels = [
    "Normal (Healthy skin)",
    "Mild Erythema",
    "Hyperpigmentation",
    "Dryness / Flaking",
    "Active Acne Lesion"
]

with open(labels_path, "w", encoding="utf-8") as f:
    for label in sample_labels:
        f.write(f"{label}\n")

print(f"✅ Labels saved to: {labels_path}")

In [ ]:
# Step 7: Compression Ratio Analysis & Visualization
reduction_pct = (1.0 - (int8_size_mb / fp32_size_mb)) * 100
compression_factor = fp32_size_mb / int8_size_mb

print(f"--- Quantization Results Summary ---")
print(f"Original (Float32):  {fp32_size_mb:.2f} MB")
print(f"Quantized (INT8):    {int8_size_mb:.2f} MB")
print(f"Size Reduction:      {reduction_pct:.1f}%")
print(f"Compression Factor:  {compression_factor:.1f}x smaller")

# Plotting the chart for presentation slides
fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
models = ['Original (float32)', 'Compressed (int8)']
sizes = [fp32_size_mb, int8_size_mb]
colors = ['#38BDF8', '#0F766E']

bars = ax.bar(models, sizes, color=colors, width=0.5)
ax.set_ylabel('Model Size (MB)', fontsize=11, fontweight='bold')
ax.set_title('On-Device Compression: Float32 vs INT8', fontsize=12, fontweight='bold', pad=12)
ax.set_ylim(0, max(sizes) * 1.25)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.4, f"{yval:.2f} MB", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('quantization_comparison.png')
plt.show()

In [ ]:
# Step 8: Automatically download files when running in Google Colab
try:
    from google.colab import files
    print("Downloading files from Colab...")
    files.download(fp32_path)
    files.download(int8_path)
    files.download(labels_path)
    files.download("quantization_comparison.png")
except ImportError:
    print(f"Files saved locally to: {os.path.abspath(output_dir)}")